In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.table("oece.bronze.seace_records")

# Misma regla que en procesos: versión más reciente de cada ocid
w = Window.partitionBy("ocid").orderBy(
    F.to_timestamp("compiled_release.date").desc(), F.col("periodo").desc()
)
ultimos = bronze.withColumn("rn", F.row_number().over(w)).filter("rn = 1")

df_adj = (
    ultimos
    .select("ocid", F.explode("compiled_release.awards").alias("a"), "_fecha_ingesta")
    .select("ocid", "a", F.explode("a.suppliers").alias("s"), "_fecha_ingesta")
    .select(
        "ocid",
        F.col("a.id").alias("id_adjudicacion"),
        F.to_date(F.to_timestamp("a.date")).alias("fecha_adjudicacion"),
        F.col("a.value.amount").cast("decimal(18,2)").alias("monto_adjudicado"),
        F.col("a.value.currency").alias("moneda"),
        F.col("s.id").alias("id_proveedor"),
        F.regexp_replace("s.id", "^PE-RUC-", "").alias("ruc_proveedor"),
        F.col("s.name").alias("proveedor"),
        F.size("a.items").alias("num_items"),
        "_fecha_ingesta",
    )
    .dropDuplicates(["id_adjudicacion", "id_proveedor"])
)

(df_adj.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("oece.silver.adjudicaciones"))

display(
    spark.table("oece.silver.adjudicaciones").alias("a")
    .join(spark.table("oece.silver.procesos").alias("p"), "ocid")
    .groupBy("categoria")
    .agg(F.count("*").alias("adjudicaciones"), F.sum("monto_adjudicado").alias("monto_total"))
)

categoria,adjudicaciones,monto_total
goods,5381,2468844940.78
works,2359,8228972990.05
services,5421,3051058929.32


In [0]:
display(
    spark.table("oece.silver.adjudicaciones")
    .groupBy("moneda")
    .agg(
        F.count("*").alias("adjudicaciones"),
        F.sum("monto_adjudicado").alias("monto_total"),
        F.max("monto_adjudicado").alias("monto_maximo"),
    )
    .orderBy(F.desc("adjudicaciones"))
)

moneda,adjudicaciones,monto_total,monto_maximo
PEN,12936,13592718587.96,779857054.80
USD,206,114907664.73,33900000.00
EUR,17,41240194.95,40410616.89
GBP,2,10412.51,5541.83


In [0]:
adj = spark.table("oece.silver.adjudicaciones").drop("tipo_cambio", "monto_adjudicado_pen")
tc = spark.table("oece.silver.tipo_cambio")

tc_mes = tc.select("moneda", "periodo", F.col("tipo_cambio").alias("tc_mes"))
# Respaldo: promedio por moneda, para meses sin tasa propia
tc_moneda = tc.groupBy("moneda").agg(F.round(F.avg("tipo_cambio"), 4).alias("tc_moneda"))

df_pen = (
    adj
    .withColumn("periodo", F.date_format("fecha_adjudicacion", "yyyy-MM"))
    .join(tc_mes, ["moneda", "periodo"], "left")
    .join(tc_moneda, "moneda", "left")
    .withColumn(
        "tipo_cambio",
        F.when(F.col("moneda") == "PEN", F.lit(1.0)).otherwise(F.coalesce("tc_mes", "tc_moneda")),
    )
    .withColumn("monto_adjudicado_pen", (F.col("monto_adjudicado") * F.col("tipo_cambio")).cast("decimal(18,2)"))
    .drop("periodo", "tc_mes", "tc_moneda")
)

(df_pen.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("oece.silver.adjudicaciones"))

display(
    spark.table("oece.silver.adjudicaciones")
    .groupBy("moneda")
    .agg(
        F.count("*").alias("adjudicaciones"),
        F.sum("monto_adjudicado_pen").alias("monto_total_pen"),
        F.sum(F.col("monto_adjudicado_pen").isNull().cast("int")).alias("sin_convertir"),
    )
)

moneda,adjudicaciones,monto_total_pen,sin_convertir
PEN,12936,13592718587.96,0
USD,206,395579105.87,0
GBP,2,48511.89,0
EUR,17,159806416.87,0
